# NB11 – Publish + dataset card (v2)

Writes a full dataset card (`README.md`), optionally flips the repo public,
and tags a version. GPU not needed.

In [ ]:
import importlib.util, sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
for _m, _p in [("huggingface_hub","huggingface_hub>=0.23"),
               ("pyarrow","pyarrow"), ("PIL","pillow"), ("datasets","datasets")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all base deps present")

In [ ]:
REPO_ID    = "Shanmuk4622/ai-detection-dataset-v2"
MAKE_PUBLIC = False   # set True after NB10 passes
VERSION_TAG = "v2.0"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")

In [ ]:
# gather final counts
try:
    mp  = C.hf_hub_download(REPO_ID, "manifest.parquet", repo_type="dataset", token=TOKEN)
    man = C.pq.read_table(mp, columns=["generator","label","split"])
    from collections import Counter
    by_gen   = Counter(man.column("generator").to_pylist())
    by_split = Counter(man.column("split").to_pylist())
    n_real   = sum(1 for l in man.column("label").to_pylist() if l==0)
    n_ai     = sum(1 for l in man.column("label").to_pylist() if l==1)
except Exception as e:
    print("manifest not ready, counting shards:", e)
    by_gen   = {m: C.count_rows(REPO_ID, f"data/{m}/", TOKEN) for m in cfg["generators"]}
    n_real   = C.count_rows(REPO_ID, "real/", TOKEN)
    n_ai     = sum(by_gen.values()); by_split = {}
print(f"real={n_real} | ai={n_ai} | by_gen={dict(by_gen)} | splits={dict(by_split)}")

In [ ]:
# build the dataset card (README.md)
gens = cfg["generators"]
gens_md = "\n".join(
    f"| `{k}` | `{v['model_id']}` | {v['native']} | {v['steps']} | {v['guidance']} |"
    for k,v in gens.items())

sp = cfg["split"]
readme = f"""---
license: other
task_categories:
- image-classification
language:
- en
tags:
- ai-generated-image-detection
- synthetic-image-detection
- diffusion-models
pretty_name: AI-Generated Image Detection Dataset v2
size_categories:
- 10K<n<100K
---

# AI-Generated Image Detection Dataset v2

Paired real / AI images for training and evaluating AI-generated image detectors.
Every real image has **one AI partner per generator**, and the pair shares a single
image-grounded caption — so detectors are pushed toward the synthesis fingerprint
(texture, frequency, rendering artefacts) rather than scene content.

## At a glance

| | |
|---|---|
| Real images | {n_real:,} (COCO + ImageNet-1k), label `0` |
| AI images | {n_ai:,} across {len(gens)} generators, label `1` |
| Pairs | {n_real:,} (each real has 6 AI partners) |
| Resolution | {cfg['target_resolution']}×{cfg['target_resolution']} RGB PNG |
| Preprocessing | Identical canonical pipeline for real **and** AI (`pipeline_version={cfg['pipeline_version']}`) |
| Master seed | `{MASTER_SEED}` — controls generation seeds and split assignment |
| Split | {int(sp['train']*100)}/{int(sp['val']*100)}/{int(sp['test']*100)} pair-level train/val/test |

## Generators

| key | model id | native res | steps | guidance |
|-----|----------|-----------|-------|----------|
{gens_md}

## Pairing and captions

Real images were captioned with **{cfg['caption_model']}**, cleaned and capped to
{cfg['caption_max_tokens']} CLIP tokens, then reused byte-identically by all six generators.
`source_real_id` links every AI image back to its real partner (`source_real_id == image_id`
for real rows).

## Canonical preprocess (leak-free)

Both real and AI images go through the **same** pipeline:
1. EXIF-transpose → convert to RGB → center-crop square → Lanczos resize to 512
2. JPEG-equalise (quality 95, 4:4:4) → PNG (compress level 6, no EXIF/ICC)

Applying an identical transform to both classes eliminates all resolution, colour-space,
and JPEG-artefact shortcuts that would let a model cheat.

## Splits

Deterministic **pair-level** split keyed on `source_real_id` (seed `{sp['seed']}`):
every real image and all six of its AI partners land in the same fold.
The split map is in `splits.parquet`; the full index is in `manifest.parquet`.

| Split | Images |
|-------|--------|
| train | {by_split.get('train', '—')} |
| val   | {by_split.get('val',   '—')} |
| test  | {by_split.get('test',  '—')} |

## Loading

```python
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
from PIL import Image
import io

# Read the manifest (no image bytes — fast)
man = pq.read_table(hf_hub_download("{REPO_ID}", "manifest.parquet", repo_type="dataset"))

# Read image bytes from a shard
shard = hf_hub_download("{REPO_ID}", "real/real-xxxxx-00000.parquet", repo_type="dataset")
tbl   = pq.read_table(shard)
img   = Image.open(io.BytesIO(tbl["image"][0].as_py()))
```

## Schema

| column | type | description |
|--------|------|-------------|
| `image_id` | string | globally unique, e.g. `real_000001` / `sdxl_000001` |
| `source_real_id` | string | pairing key; equals `image_id` for real rows |
| `label` | int8 | 0 = real, 1 = AI |
| `generator` | string | `real` or `sd15` or `sdxl` or `flux_schnell` or `kandinsky22` or `pixart_sigma` or `sd_cascade` |
| `source_dataset` | string | `coco` or `imagenet` or `<generator>` |
| `prompt` | string | caption used (AI rows); null for real |
| `image` | binary | canonical 512×512 RGB PNG bytes |
| `width` / `height` | int16 | always 512 |
| `orig_width` / `orig_height` | int32 | provenance only — **never use as a training feature** |
| `gen_model_id` | string | HF model id |
| `gen_steps` / `gen_guidance` / `gen_native_res` | int16/float32/int16 | generation params |
| `seed` | int64 | `hash(master_seed, generator, source_real_id)` |
| `caption_model` | string | captioner used |
| `pipeline_version` | string | `{cfg['pipeline_version']}` |
| `sha256` | string | hex digest of the PNG bytes |
| `created_utc` | string | ISO-8601 UTC timestamp |

## Provenance, license and intended use

- **ImageNet** content is **non-commercial research only** (ILSVRC terms).
- **COCO** images are Flickr-sourced (Creative Commons).
- AI images are synthetic outputs of the models listed above, each under its own license.
- This dataset inherits the most restrictive applicable term: **non-commercial research use**.
  Not legal advice.
- Intended for training / evaluating AI-image detectors.
  See `validation_report.json` for the leak-audit results.

## Limitations and caveats

- Core set is **text-to-image** only (no img2img / reference conditioning).
  Detectors trained here target the text-to-image threat model.
- Caption quality (BLIP-2 concise sentences) bounds prompt diversity.
- Stable Cascade and Flux run with CPU offload on T4, so their generation conditions
  differ slightly from a high-end GPU setup.
- Per-model step counts were reduced for speed (SDXL 8 steps, SD 1.5 20 steps);
  this is representative of real-world fast inference.
"""
open("README.md","w").write(readme)
C.HfApi(token=TOKEN).upload_file(path_or_fileobj="README.md", path_in_repo="README.md",
        repo_id=REPO_ID, repo_type="dataset", commit_message="dataset card v2")
print("README.md uploaded  (", len(readme), "chars )")

In [ ]:
api = C.HfApi(token=TOKEN)
if MAKE_PUBLIC:
    api.update_repo_visibility(repo_id=REPO_ID, private=False, repo_type="dataset")
    print("repo is now PUBLIC")
else:
    print("still private (set MAKE_PUBLIC=True when ready)")
try:
    api.create_tag(repo_id=REPO_ID, repo_type="dataset", tag=VERSION_TAG)
    print("tagged", VERSION_TAG)
except Exception as e:
    print("tag note:", e)
print("\nNB11 complete — dataset build finished.")